# 2→2→1 신경망의 순전파와 역전파 유도

고정 입력의 값을 손으로 전개한 뒤 [실제 구현](../src/backprop.py)의 NumPy 결과와 소수점 4자리까지 비교했다. 내부 계산은 반올림하지 않고 표를 표시할 때만 반올림했다.

PDF 3쪽의 `y_true`는 빈칸이므로 7쪽 예시의 **1**을 사용했다. 예시 값은 정답이 아니라 참고라고 명시되어 있다. 예시의 `z2=0.6071`, `dz1[1]=-0.0513`은 원래 가중치에서 다시 계산하면 각각 **0.6072**, **-0.0517**로 반올림된다.

## 벡터와 shape 규약

`W1`의 각 행은 은닉 뉴런 하나의 가중치다. NumPy 1차원 배열은 행/열을 고정하지 않으므로 곱셈 방향을 식에 명시했다.

| 값 | shape | 의미 |
|---|---|---|
| x=[1,0] | (2,) | 입력 두 개 |
| W1=[[.1,.2],[.3,.4]] | (2,2) | 은닉 2개 × 입력 2개 |
| b1=[0,0], z1, a1 | (2,) | 은닉 뉴런 두 개 |
| W2=[.5,.6] | (2,) | 출력 뉴런의 가중치 |
| b2=0, z2, y_pred, y_true=1, L | () | 스칼라 |

입력 두 항목을 생활 예로 바꾸면 공부 시간과 이전 정답률 같은 두 특징이다. 이 실험의 숫자는 데이터로 학습한 값이 아니라 연쇄 법칙을 확인하는 고정 예제다.

In [1]:
from pathlib import Path
import os
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ['MPLCONFIGDIR'] = str(ROOT / '.cache' / 'matplotlib')
import numpy as np
np.random.seed(42)
print('Python:', sys.executable)
print('NumPy:', np.__version__)


Python: /Users/oliverjoo/Dev/Anaconda/anaconda3/envs/py312/bin/python
NumPy: 1.26.4


## 1. 순전파를 손으로 계산

$$z_1=W_1x+b_1=\begin{bmatrix}0.1\cdot1+0.2\cdot0\\0.3\cdot1+0.4\cdot0\end{bmatrix}=\begin{bmatrix}0.1\\0.3\end{bmatrix}\quad (2,)$$
$$\sigma(z)=\frac1{1+e^{-z}},\qquad a_1=\sigma(z_1)=[0.5249791875,0.5744425168]\quad(2,)$$
$$z_2=W_2a_1+b_2=0.5(0.5249791875)+0.6(0.5744425168)=0.6071551038\quad()$$
$$\hat y=\sigma(z_2)=0.6472915700\quad()$$
$$L=-[y\ln\hat y+(1-y)\ln(1-\hat y)]=-\ln(0.6472915700)=0.4349584369\quad()$$

중간 체크포인트: `z1=[.1000,.3000]`, `a1=[.5250,.5744]`, `z2=.6072`, `y_pred=.6473`.

In [2]:
from src.backprop import fixed_example, forward_backward
example = fixed_example()
values, gradients = forward_backward(**example)
for key, value in values.items():
    print(f'{key:8s} shape={np.shape(value)} value={np.round(value, 4)}')

z1       shape=(2,) value=[0.1 0.3]
a1       shape=(2,) value=[0.525  0.5744]
z2       shape=() value=0.6072
y_pred   shape=() value=0.6473
loss     shape=() value=0.435


## 2. 출력에서 입력 방향으로 연쇄 법칙 적용

연쇄 법칙은 $dL/dz=(dL/da)(da/dz)$다. 연결된 각 단계의 변화율을 곱한다. $\sigma'(z)=\sigma(z)(1-\sigma(z))$를 사용했다.

### 출력 확률의 미분 — 스칼라 ()
$$\frac{\partial L}{\partial\hat y}=-\frac y{\hat y}+\frac{1-y}{1-\hat y}=-1.5448988468$$

### 출력층 선형값의 미분 — 스칼라 ()
$$\delta_2=\frac{\partial L}{\partial z_2}=\frac{\partial L}{\partial\hat y}\hat y(1-\hat y)=\hat y-y=-0.3527084300$$

BCE와 Sigmoid가 결합되어 $\hat y-y$로 단순해진다. 현재 정답이 1인데 예측은 .6473이므로 음의 기울기가 나온다.

### 출력 가중치의 미분 — (2,)
$$\frac{\partial L}{\partial W_2}=\delta_2 a_1=[-0.1851645850,-0.2026107182]$$

### 은닉 활성화의 미분 — (2,)
$$\frac{\partial L}{\partial a_1}=\delta_2 W_2=[-0.1763542150,-0.2116250580]$$

### 은닉 선형값의 미분 — (2,)
$$\delta_1=\frac{\partial L}{\partial z_1}=\frac{\partial L}{\partial a_1}\odot a_1\odot(1-a_1)=[-0.0439785158,-0.0517335044]$$

### 첫 가중치 행렬의 미분 — (2,2)
$$\frac{\partial L}{\partial W_1}=\delta_1x^T=\begin{bmatrix}-0.0439785158&0\\-0.0517335044&0\end{bmatrix}$$

`np.outer(dz1, x)`를 사용했다. 두 번째 입력이 0이라 두 번째 열의 미분도 0이다. 편향은 덧셈이므로 $\partial L/\partial b_1=\delta_1$ (2,), $\partial L/\partial b_2=\delta_2$ ()다.

| 필수 체크포인트 | shape | 손계산, 소수점 4자리 |
|---|---|---|
| dL/dy_pred | () | -1.5449 |
| dL/dz2 | () | -0.3527 |
| dL/dW2 | (2,) | [-0.1852,-0.2026] |
| dL/da1 | (2,) | [-0.1764,-0.2116] |
| dL/dz1 | (2,) | [-0.0440,-0.0517] |
| dL/dW1 | (2,2) | [[-0.0440,0],[-0.0517,0]] |

In [3]:
hand_forward = {'z1':[.1,.3], 'a1':[.5250,.5744], 'z2':.6072, 'y_pred':.6473, 'loss':.4350}
hand_backward = {'dy_pred':-1.5449, 'dz2':-.3527, 'dW2':[-.1852,-.2026],
                 'da1':[-.1764,-.2116], 'dz1':[-.0440,-.0517],
                 'dW1':[[-.0440,0],[-.0517,0]]}
for data, hand in [(values, hand_forward), (gradients, hand_backward)]:
    for key, expected in hand.items():
        rounded = np.round(data[key], 4)
        np.testing.assert_array_equal(rounded, np.asarray(expected))
        print(f'{key:8s} shape={np.shape(data[key])} NumPy={rounded}: 4자리 일치')

z1       shape=(2,) NumPy=[0.1 0.3]: 4자리 일치
a1       shape=(2,) NumPy=[0.525  0.5744]: 4자리 일치
z2       shape=() NumPy=0.6072: 4자리 일치
y_pred   shape=() NumPy=0.6473: 4자리 일치
loss     shape=() NumPy=0.435: 4자리 일치
dy_pred  shape=() NumPy=-1.5449: 4자리 일치
dz2      shape=() NumPy=-0.3527: 4자리 일치
dW2      shape=(2,) NumPy=[-0.1852 -0.2026]: 4자리 일치
da1      shape=(2,) NumPy=[-0.1764 -0.2116]: 4자리 일치
dz1      shape=(2,) NumPy=[-0.044  -0.0517]: 4자리 일치
dW1      shape=(2, 2) NumPy=[[-0.044  -0.    ]
 [-0.0517 -0.    ]]: 4자리 일치


## 3. 손실의 수치 미분으로 한 번 더 검증

각 가중치와 편향 하나를 $h=10^{-5}$만큼 움직여 $[L(\theta+h)-L(\theta-h)]/(2h)$를 계산했다. 손계산 표만 맞춘 구현인지 확인하기 위해 별도의 수치 미분을 사용했다. AutoGrad는 사용하지 않았다.

In [4]:
for name in ['W1','b1','W2','b2']:
    parameter = np.asarray(example[name], dtype=float)
    numerical = np.zeros_like(parameter)
    for index in np.ndindex(parameter.shape):
        plus, minus = parameter.copy(), parameter.copy()
        plus[index] += 1e-5
        minus[index] -= 1e-5
        loss_plus = forward_backward(**{**example, name:plus})[0]['loss']
        loss_minus = forward_backward(**{**example, name:minus})[0]['loss']
        numerical[index] = (loss_plus-loss_minus)/2e-5
    error = np.max(np.abs(numerical-gradients['d'+name]))
    np.testing.assert_allclose(numerical, gradients['d'+name], atol=1e-8)
    print(name, 'maximum absolute gradient error:', float(error))

W1 maximum absolute gradient error: 1.0556548690754397e-11
b1 maximum absolute gradient error: 1.0556548690754397e-11
W2 maximum absolute gradient error: 3.259587044723844e-12
b2 maximum absolute gradient error: 7.178257988016412e-12


순전파는 예측값을 만들고, 역전파는 같은 계산 경로를 거꾸로 따라 각 파라미터의 미분을 구한다. 이 노트북의 범위는 고정 예제의 유도와 검증까지다. 모델 학습 루프를 추가하지 않았다.